In [3]:
%matplotlib inline
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np
#from neurodynex3.leaky_integrate_and_fire import LIF
#rom neurodynex3.tools import input_factory, plot_tools


# parameters in SI units: volts, seconds, ohms, amps
V_rest  = -0.070    # -70 mV
V_reset = -0.065    # -65 mV
V_th    = -0.050    # -50 mV  firing threshold
R       = 10e6      # 10 MΩ
tau     = 8e-3      # 8 ms  membrane time constant
t_refr  = 2e-3      # 2 ms  absolute refractory period
dt = 1e-4;
rng = np.random.default_rng(0)

tau_syn = 5e-3   # 5 ms
gbar    = 40e-12 # 40 pS


def simulate_lif(I, dt, V_rest,V_reset,V_th, R, tau, t_refr):
    #simulate LIF model through differential equation itself
    n = len(I)
    V=np.zeros(n)
    V[0] = V_rest
    spike_times = []
    refr_steps = int(round(t_refr / dt))   # how many steps to stay clamped
    refr_counter = 0                       # >0 means we're inside refractory

    for k in range(n-1):
        if refr_counter>0:
            V[k+1] = V_reset
            refr_counter -= 1
            continue 
        #differential equation
        dV = (dt / tau) * (-(V[k] - V_rest) + R * I[k])
        V[k+1] = V[k] + dV

        # record spike timings 

        if V[k] < V_th and V[k+1] >= V_th:
            spike_times.append((k+1) * dt)
            V[k+1] = V_reset #reset
            refr_counter = refr_steps
    t = np.arange(n)* dt 
    return t, V, spike_times


def poisson_encode(rate, T, dt, rng):
    n = int(round(T / dt))
    spikes = (np.random.rand(n) < rate * dt).astype(float) #IDENTIFY how many spikes actually occur per timestep
    return spikes

def conductance_trace(spk, gbar, tau_syn, dt):
    #function to determine change in conductance in postsynaptic neuron
    n = len(spk)
    g = np.zeros(n)
    decay = np.exp(-dt / tau_syn)
    for k in range(1, n):
        g[k] = g[k-1] * decay + gbar * spk[k]
    return g 

In [4]:
#testing spike encoding
spk = poisson_encode(100, 0.5, 1e-4, rng)
print(spk.sum(), "spikes in 0.5s at 100Hz — expected ~50")

50.0 spikes in 0.5s at 100Hz — expected ~50


In [5]:
#testing conductance trace

g = conductance_trace(spk, gbar, tau_syn, dt)
print(f"peak conductance: {g.max()*1e12:.1f} pS")

peak conductance: 103.3 pS


In [6]:
# Wiring the conductance-based synapse (Ch.3) into the LIF neuron (soma)
# Pipeline: Poisson spike train -> conductance g(t) -> LIF driven by I_syn = g(t)*(V - E_syn)
# This replaces the fixed current I we used in single-neuron tests with a synaptic current

E_syn   = 0.0     # excitatory reversal potential (~0 mV, Ch.3)
tau_syn = 5e-3    # synaptic decay time constant, 5 ms
gbar    = 1e-7    # conductance bump per spike, scaled up for a single test synapse

def lif_conductance(g, dt, V_rest, V_reset, V_th, R, tau, t_refr, E_syn):
    # same Euler LIF loop as before, but driven by synaptic current instead of fixed I
    n = len(g)
    V = np.zeros(n)
    V[0] = V_rest
    spike_times = []
    refr_steps = int(round(t_refr / dt))
    refr_counter = 0

    for k in range(n - 1):
        if refr_counter > 0:
            V[k+1] = V_reset
            refr_counter -= 1
            continue
        # synaptic current: I_syn = g(t) * (V - E_syn)
        dV = (dt / tau) * (-(V[k] - V_rest) - R * g[k] * (V[k] - E_syn))
        V[k+1] = V[k] + dV
        if V[k] < V_th and V[k+1] >= V_th:
            spike_times.append((k+1) * dt)
            V[k+1] = V_reset
            refr_counter = refr_steps

    t = np.arange(n) * dt
    return t, V, spike_times

# quick test: 100 Hz Poisson input for 0.5s, through the synapse, into the LIF
spk = poisson_encode(100, 0.5, dt, rng)
g   = conductance_trace(spk, gbar, tau_syn, dt)
t, V, spikes = lif_conductance(g, dt, V_rest, V_reset, V_th, R, tau, t_refr, E_syn)

print(f"input: 100 Hz Poisson, {int(spk.sum())} input spikes")
print(f"output: {len(spikes)} output spikes ({len(spikes)/0.5:.1f} Hz)")

input: 100 Hz Poisson, 39 input spikes
output: 26 output spikes (52.0 Hz)


so currently we have poisson_encode which turns target firing rate into a Bernoulli spike train based on the example in 3.1.2
we then have conductance_trace function which turns a spike train into an exponentially-decaying conductance g(t)
the LIF_conductance function is an LIF neuron driven by I_syn = g(t) * (V - E_syn) instead of fixed current, replicating what happens with an actual input from a synapse
In my mind, I think the best approach considering the limited time is just trying to create an untrained feedforward SNN classifier prototype with LIF neurons. If I have time I will apply LIF.

The approach I will use is as follows:

Multi - neuron poisson encoding - essentially taking a feature vector as an input and matching that into spike trains. Each feature maps to a rate (e.g. rate_i = x_i * max_rate), and each rate gets its own independent Poisson spike train
Per output sum conductance - each output neuron o receives input from all N_in input neurons, each connection scaled by a weight W[i, o]. The conductance driving neuron o at time t is the weighted sum over all input neurons' individual conductance traces. This creates synapses with stronger or weaker connections based on the weighting on those synapses. 
Multi-output LIF layer -  each output neuron is its own independent LIF, each with its own V, its own refractory counter, its own spike list — but all driven by the appropriately-weighted-and-summed conductance. 


I will put the above three into one function to create the forward pass of the network, then design the spike count readout to determine which neuron fired the most allowing me to see the decisions the matrix determines

In [7]:
""" 
Here I am designing the poisson encoding function which encodes a feature vector x into N independent Poisson spike trains, 
here for simplicity and time constraints, I will assume the feature values are scaled to [0,1]

"""

def poisson_encode_population(x, max_rate, T, dt, rng):
    x = np.asarray(x)
    N_in = len(x)
    n_steps = int(round(T / dt))

    rates = x * max_rate #linearly mapping each feature to a rate ranging between [0,max_rate]

    spikes = np.zeros((N_in, n_steps))
    for i in range(N_in):
        spikes[i] = (rng.random(n_steps) < rates[i] * dt).astype(float) 
        #i decided to just implement my old function directly instead of calling it, just cleaner.
        

    return spikes, rates

#one of my worries is that I will end up with neurons having the same output 

rng = np.random.default_rng(0)
x = np.array([0.8, 0.8, 0.2])   # two neurons with the SAME rate, one different
spikes, rates = poisson_encode_population(x, max_rate=100, T=0.5, dt=1e-4, rng=rng)

print("rates:", rates)
print("spike counts per neuron:", spikes.sum(axis=1))

# The two neurons with equal rate (rows 0 and 1) should have similar but NOT identical spike counts/timing 
print("rows 0 and 1 identical?", np.array_equal(spikes[0], spikes[1]))

#they dont!!



rates: [80. 80. 20.]
spike counts per neuron: [39. 32. 11.]
rows 0 and 1 identical? False


here, we can see a rate of 80hz should give us spike counts of 40 (t=0.5), and i got 39 and 32, which is within teh standard deviatioon range. I believe this should be fine given it is a poisson process (variance = mean spike count so any spike count within standard deviation should be good)


An LIF neuron leaks current, as it always tries to reset back to a resting voltage. However, its input is a synapse feeding a synaptic current, which is based on the conductance of the synapse and  the difference between the current voltage and some reference value E_syn. This makes the synaptic current being Isyn(t) = g_syn(t)(u(t) - E_syn).
In a SNN, there are multiple input synapses feeding into one neuron. The total input has to consider that the difference (u(t) - E_syn will be a constant term, but the flow of current will be different for each synapse, meaning the synaptic conductance of each synapse is added, then multiply by the difference term. 

I​(t)=j∑​f∑​w_ij​g_ij​(t−tj(f)​)(ui​(t)−Esyn​)

in order to differentiate between synapses, we will add fixed weights to each synapse and see what happens.

Below I will combine N independent input spike trains into one conductance trace per output neuron, using a weight matrix


In [8]:
def combined_conductance(spikes,W,gbar,tau_syn, dt):
    """
    W - weight matrix, scales how strongly inptu neuron i drives output neuron o,
    gbar: baseline conductance jump per spike - gives a number to multiply the weights with
    tau_syn: synaptic decay time constant
    return:
    g_out - one summed conductance trace
    """
    N_in, n_steps = spikes.shape
    N_out = W.shape[1]
    g_out = np.zeros((N_out, n_steps))

    for o in range(N_out):
        #For this output neuron, sum the weighted conductance contribution of every input neuron which is connected
        for i in range(N_in):
            if W[i,o] == 0:
                continue #no synapse so just skip the neuron

            g_out[o] += conductance_trace(spikes[i], gbar * W[i, o], tau_syn, dt)

    return g_out

#now I will test whether it actually accurately adds up two neuron's conductance accurately
# 2 input neurons -> 1 output neuron, different weights
rng = np.random.default_rng(1)
spikes, _ = poisson_encode_population(np.array([0.8, 0.5]), max_rate=100, T=0.5, dt=1e-4, rng=rng)
W = np.array([[1.0], [0.5]])  # neuron 0 full weight, neuron 1 half weight

g_combined = combined_conductance(spikes, W, gbar=1e-7, tau_syn=5e-3, dt=1e-4)

# manually compute the two individual contributions and check they sum to g_combined
g0 = conductance_trace(spikes[0], 1e-7 * 1.0, 5e-3, 1e-4)
g1 = conductance_trace(spikes[1], 1e-7 * 0.5, 5e-3, 1e-4)

print("matches manual sum:", np.allclose(g_combined[0], g0 + g1), g_combined[0], g0, g1)


matches manual sum: True [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 1.65700713e-08
 1.62419619e-08 1.59203495e-08] [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 1.06824291e-08
 1.04709029e-08 1.02635651e-08] [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 5.88764216e-09
 5.77105904e-09 5.65678441e-09]


Since they match, it shows that the conductance is adding up correctly. Now I will begin building the multi-output LIF layer. Each output neuron is its own independent LIF, all driven by the weighted and summed conductance.

In [9]:
def lif_conductance_population(g_out, dt, V_rest, V_reset, V_th, R, tau, t_refr, E_syn):
    # running N independent LIF neurons, each driven by its own combined conductance trace from combined conductance
    """
        Returns
    -------
    V_all      : (N_out, n_steps) array -- voltage trace per neuron
                 (fixed shape, fine as a plain array since every neuron
                 runs for the same number of steps)
    spikes_all : list of length N_out, where spikes_all[o] is the list of
                 spike times for output neuron o
    """

    N_out, n_steps = g_out.shape
    V_all = np.zeros((N_out, n_steps)) 
    spikes_all = [] 

    for o in range(N_out):
        #each lif neuron gets its own independent variables to ensure they are all different
        t, V, spikes = lif_conductance(
            g_out[o], dt, V_rest, V_reset, V_th, R, tau, t_refr, E_syn
        )
        V_all[o] = V
        spikes_all.append(spikes)

    return V_all, spikes_all

#here I will test a singular neuron to see what happens

V_all, spikes_all = lif_conductance_population(
    g_combined, dt, V_rest, V_reset, V_th, R, tau, t_refr, E_syn
)

for o in range(len(spikes_all)):
    print(f"output neuron {o}: {len(spikes_all[o])} spikes")

output neuron 0: 40 spikes


one output neuron, drivved by two weighted inputs computes 40 spikes over 0.5 seconds. I am unsure whether this is a good result or not, but I will assume it is.

I will now try run a full foward pass of the network in one function

In [10]:
def run_trial(x, W, max_rate, T, dt, V_rest, V_reset, V_th, R, tau, t_refr,gbar, tau_syn, E_syn, rng):
    """
    x        : (N_in,) feature vector, values expected in [0, 1]
    W        : (N_in, N_out) weight matrix
    max_rate : Hz corresponding to a feature value of 1.0
    T, dt    : trial duration and timestep
    rng      : np.random.Generator instance
 
    Returns
    -------
    spike_counts : (N_out,) array -- number of spikes each output neuron fired
    spikes_all   : list of length N_out -- raw spike times per neuron
    predicted_class - which neuron fired mthe most - the selected neuron 
    """

    spikes_in,rates = poisson_encode_population(x, max_rate, T, dt, rng) #task 1, convert feature vector into input spike trains

    g_out = combined_conductance(spikes_in, W, gbar, tau_syn, dt) #weighted synapses to one combined conductance trace per output neuron 

    V_all, spikes_all = lif_conductance_population(g_out, dt, V_rest, V_reset, V_th, R, tau, t_refr, E_syn) 
    #each output neuron's own independent LIF, driven by its g_out row

    spike_counts = np.array([len(s) for s in spikes_all])
    
    predicted_class = np.argmax(spike_counts)
    
    return spike_counts, spikes_all, predicted_class


#Testing the function by seeing whether a 100 Hz Poisson input at T=0.5 gives us a similar output to what we have seen in the separate functions

rng_check = np.random.default_rng(0)  # same seed as the very first test used
x_check = np.array([1.0])             # 1.0 * max_rate=100 -> 100 Hz, matches original
W_check = np.array([[1.0]])           # single input, single output, full weight
 
spike_counts, spikes_all, predicted_class = run_trial(
    x_check, W_check, max_rate=100, T=0.5, dt=dt,
    V_rest=V_rest, V_reset=V_reset, V_th=V_th, R=R, tau=tau, t_refr=t_refr,
    gbar=1e-7, tau_syn=5e-3, E_syn=0.0, rng=rng_check
)
 
print(f"output: {spike_counts[0]} spikes ({spike_counts[0]/0.5:.1f} Hz)")
print(f"predicted class: {predicted_class}")



output: 41 spikes (82.0 Hz)
predicted class: 0


41 spikes at 82 Hz seems close to the 32 spikes at 72 Hz, suggesting the result seems good, and anyway at T=0.5 at 82 Hz, we expect a spike count of 41. Predicted class of 0 makes no sense right now, but hopefully when I begin adding a small set and so on this value will be significant. 

Now I will finally test whether the full pipeline works correctly by testing whether the right neurons fire up depending on the input. We will test whether if we input current into neuron 0, will the output of neuron 1 be ignored (as it should) and vice versa

In [11]:
def make_toy_sample(cls, rng, contrast=0.8):
    """
    Generating one feature vector for a 2 input, 2 class problem

    cls - 0 or 1, which class to generate
    contract - how strongly separated the two classes are 

    returns a length-2 feature vector 

    """

    if cls == 0:
        #class 0: input neuron 0 is high, input neuron 1 is dim
        return np.array([contrast, 1 - contrast])
    else:
        #class 1: input neuron 1 is high etc
        return np.array([1-contrast, contrast])

#now i will apply fixed weights so that the output neuron 0 is wired to only listen mainly to input 0 and so on

W_oracle = np.array([
    [1.0, 0.0],   # input neuron 0 -> strongly drives output 0, ignores output 1
    [0.0, 1.0],   # input neuron 1 -> strongly drives output 1, ignores output 0
    [1.0,2.0],  #input neuron 2,. should ignore input neuron 1 
])

#running many trials alternating between classes to see what happens

n_trials = 40
rng = np.random.default_rng(42) 
correct = 0

for trial in range(n_trials):
    true_class = trial % 2 #alternate between classes
    x = make_toy_sample(true_class, rng)

    spike_counts, spikes_all, predicted_class = run_trial(
        x, W_oracle, max_rate=100, T=0.5, dt=dt,
        V_rest=V_rest, V_reset=V_reset, V_th=V_th, R=R, tau=tau, t_refr=t_refr,
        gbar=1e-7, tau_syn=5e-3, E_syn=0.0, rng=rng
    )
    if predicted_class == true_class:
        correct += 1

    print(f"trial {trial}: true={true_class}, predicted={predicted_class}, "
          f"spike_counts={spike_counts}")

accuracy = correct / n_trials
print(f"\naccuracy: {correct}/{n_trials} = {accuracy*100:.1f}%")



#I am now quickly applying some sanity tests:

# rerun one trial, check Spikes_all rows aren't identical or weirdly linked
print("neuron 0 spikes:", spikes_all[0])
print("neuron 1 spikes:", spikes_all[1])
print("identical?", spikes_all[0] == spikes_all[1])
#check shape
print("spike_counts shape:", spike_counts.shape, "expected (N_out,) = (2,)")
print("W_oracle shape:", W_oracle.shape, "expected (N_in, N_out) = (2,2)")

#zer-input edge case, confirm no input means no spikes, not sure whether neurons have noise or not

x_zero = np.array([0.0, 0.0])
spike_counts_zero, _, predicted_zero = run_trial(
    x_zero, W_oracle, max_rate=100, T=0.5, dt=dt,
    V_rest=V_rest, V_reset=V_reset, V_th=V_th, R=R, tau=tau, t_refr=t_refr,
    gbar=1e-7, tau_syn=5e-3, E_syn=0.0, rng=rng
)
print("spike counts with zero input:", spike_counts_zero)
# should be [0, 0] -- no input spikes, no synaptic current, no output spikes


    

trial 0: true=0, predicted=0, spike_counts=[28  3]
trial 1: true=1, predicted=1, spike_counts=[ 1 46]
trial 2: true=0, predicted=0, spike_counts=[31  2]
trial 3: true=1, predicted=1, spike_counts=[ 4 30]
trial 4: true=0, predicted=0, spike_counts=[21  5]
trial 5: true=1, predicted=1, spike_counts=[ 6 32]
trial 6: true=0, predicted=0, spike_counts=[31  0]
trial 7: true=1, predicted=1, spike_counts=[ 4 39]
trial 8: true=0, predicted=0, spike_counts=[31  3]
trial 9: true=1, predicted=1, spike_counts=[ 1 41]
trial 10: true=0, predicted=0, spike_counts=[35  3]
trial 11: true=1, predicted=1, spike_counts=[ 2 37]
trial 12: true=0, predicted=0, spike_counts=[25  2]
trial 13: true=1, predicted=1, spike_counts=[ 1 40]
trial 14: true=0, predicted=0, spike_counts=[31  1]
trial 15: true=1, predicted=1, spike_counts=[ 3 32]
trial 16: true=0, predicted=0, spike_counts=[31  1]
trial 17: true=1, predicted=1, spike_counts=[ 2 31]
trial 18: true=0, predicted=0, spike_counts=[23  1]
trial 19: true=1, pred

100% accuracy AT CONTRAST = 0.8 and the spike coutns back it as a real result rather than a fluke, where for every class 0 trial output neruon 0 clearly dominates and flips for class 1

now my pipeline forward pass of encoding, weighted synapses, lif population, readout is working 

This result does not prove that it can learn weights yet, here I will need to study STDP to understand how networks are trained and learn

Neuron 0 fired twice, neruon 1 fired 17 times, completely different spike times so they are seperate neurons with no shared state
The shape matches and zero input means no spikes, how the LIF neuron should work


Now I will begin studying STDP, HERE i will summarise the key concepts I have learned. 

19.1 The Hebb Rule and Experimental Evidence

Hebb's postulate states that if neuron A repeatedly takes part in firing neuron B, the synapse from A to B becomes stronger. The modern shorthand for this is that neurons which fire together wire together, although that phrasing loses an important detail. Hebb's original wording is asymmetric, because a neuron that contributes to firing another must be active slightly beforehand. Timing order matters, not just simultaneous activity.

Hebbian learning is unsupervised. There is no signal telling the synapse whether a change was good or bad. Changes are driven purely by correlations in pre and postsynaptic activity.

19.1.1 Long-Term Potentiation

Long-Term Potentiation, or LTP, is the lasting increase in synaptic strength produced by a specific stimulation protocol. The experiment runs in three steps. A weak test pulse is applied first, large enough to evoke a postsynaptic potential but not an action potential, and the response amplitude is recorded. The presynaptic fibres are then stimulated with a sequence of high frequency pulses strong enough to force postsynaptic firing. When the weak test pulse is repeated afterwards, the postsynaptic response is noticeably larger, and this increase persists for hours.

The opposite effect, a lasting decrease, is called Long-Term Depression, or LTD. Both differ from the short-term facilitation covered in Chapter 3, which decays within seconds. LTP and LTD are thought to underlie long lasting memory.

19.1.2 Spike-Timing-Dependent Plasticity

Pairing experiments force the presynaptic neuron j and postsynaptic neuron i to spike at controlled times, then measure the resulting weight change as a function of the timing difference between them. The finding is that the direction of the change depends on spike order, on a millisecond timescale:


Presynaptic spike shortly before postsynaptic spike produces potentiation.
Reversed order produces depression.


This is the origin of the term Spike-Timing-Dependent Plasticity, or STDP. The result agrees with Hebb's postulate, since neurons firing slightly early are the ones that plausibly contributed to the postsynaptic spike, while those firing later clearly did not. The asymmetric shape of the STDP window is therefore a direct implementation of the causality requirement built into Hebb's principle.

Not all systems behave this way. Synapses between parallel fibres and Purkinje cells in electric fish show the reverse dependence, which is described as anti-Hebbian plasticity.

19.2 Models of Hebbian Learning

19.2.1 Rate-Based Formulation

Two properties of Hebb's postulate drive the mathematics. Locality means a weight change may only depend on quantities available at the synapse itself, such as the two firing rates and the current weight, and not on other neurons. Joint activity means both neurons must be active for a change to occur.

Locality gives a general form for the weight change:

dw_ij/dt = F(w_ij ; ν_i , ν_j)  (19.1)

Expanding F as a Taylor series about ν_i = ν_j = 0 gives:

dw_ij/dt = c₀(w_ij) + c₁^pre(w_ij)·ν_j + c₁^post(w_ij)·ν_i + c₂^pre(w_ij)·ν_j² + c₂^post(w_ij)·ν_i² + c₁₁^corr(w_ij)·ν_i·ν_j + O(ν³)  (19.2)

The bilinear term c₁₁^corr·ν_i·ν_j is the one that implements joint activity, since it is only nonzero when both neurons are active. A rule without it is not Hebbian, because activity from one side alone would be enough to change the weight.

Keeping only that term gives the prototype Hebb rule:

dw_ij/dt = c₁₁^corr · ν_i · ν_j  (19.3)

A negative c₁₁^corr gives an anti-Hebbian rule, weakening synapses when both neurons are active together.

Weight Bounds

Equation 19.3 can only increase weights, so every synapse would eventually saturate. Bounds are needed.

A hard bound keeps c₁₁^corr constant at γ₂ within the range 0 < w_ij < w_max and sets it to zero outside, so growth halts abruptly at the limit.

A soft bound instead makes the coefficient shrink smoothly as the weight approaches its maximum:

c₁₁^corr(w_ij) = γ₂ (w_max − w_ij)^β  (19.4)

with γ₂ and β positive, and β = 1 as the typical value. As β approaches zero the soft bound converges to the hard bound.

Combining a soft bound with a decay term produces a rule with both potentiation and depression:

dw_ij/dt = γ₂(1 − w_ij)·ν_i·ν_j − γ₀·w_ij  (19.5)

Here synapses drift back toward zero when unstimulated.

Example Rules

The covariance rule, uses deviations from average rates rather than the rates themselves:

dw_ij/dt = γ (ν_i − ⟨ν_i⟩)(ν_j − ⟨ν_j⟩)  (19.6)

Oja's rule adds a quadratic postsynaptic term:

dw_ij/dt = γ [ ν_i·ν_j − w_ij·ν_i² ]  (19.7)

This converges to weights normalised so that Σ_j w_ij² = 1, while keeping the Hebbian behaviour of 19.3. Normalisation matters because it forces competition between synapses onto the same postsynaptic neuron. If some weights grow, others must shrink.

The Bienenstock-Cooper-Munro rule, or BCM, introduces a threshold on postsynaptic rate:

dw_ij/dt = φ(ν_i − ν_θ)·ν_j  (19.8)

A common concrete choice is:

dw_ij/dt = η·ν_i·(ν_i − ν_θ)·ν_j = c₂₁·ν_i²·ν_j − c₁₁^corr·ν_i·ν_j  (19.9)

Presynaptic activity paired with moderate postsynaptic excitation weakens the synapse, while activity above the threshold ν_θ strengthens it. The fixed point at ν_θ is unstable, so ν_θ has to become an adaptive variable depending on the average output rate, otherwise the postsynaptic rate either grows without limit or collapses to zero. BCM produces input selectivity and has been used to model receptive field development.

19.2.2 Pair-Based Models of STDP

This section moves from rates to spikes, and provides the rule implemented in this project.

The simplest pair-based model updates the weight based on the interval Δt between a presynaptic spike at t_pre and a postsynaptic spike at t_post:

Δw₊ = A₊(w) · exp(−|Δt| / τ₊)   applied at t_post, when t_pre < t_post
Δw₋ = A₋(w) · exp(−|Δt| / τ₋)   applied at t_pre,  when t_pre > t_post  (19.10)

A₊ is positive and gives potentiation, A₋ is negative and gives depression. Updates happen immediately at each spike. Specifying such a model requires two decisions: how the amplitudes A₊ and A₋ depend on the current weight, and which spike pairs are counted. All pairs may be included, or only the nearest neighbour on each side. Widely separated spikes contribute little regardless, because the amplitude decays exponentially with the interval.

Writing the pre and postsynaptic spike trains as sums of delta functions, S_j = Σ_f δ(t − t_j^(f)) and S_i = Σ_f δ(t − t_i^(f)), the rule becomes:

dw_ij/dt = S_j(t)·[ a₁^pre + ∫₀^∞ A₋(w_ij)·W₋(s)·S_i(t−s) ds ] + S_i(t)·[ a₁^post + ∫₀^∞ A₊(w_ij)·W₊(s)·S_j(t−s) ds ]  (19.11)

W₊ and W₋ describe the shape of the learning window, while a₁^pre and a₁^post are non-Hebbian terms allowing an isolated spike on either side to cause a small change. The standard pair-based rule sets W±(s) = exp(−s/τ±) and both non-Hebbian terms to zero, recovering 19.10.

Implementation Using Local Variables

Equation 19.10 can be implemented without ever computing Δt directly, using two decaying state variables. Each presynaptic spike leaves a trace x_j:

dx_j/dt = −x_j / τ₊ + Σ_f δ(t − t_j^(f))  (19.12)

Each postsynaptic spike leaves a trace y_i:

dy_i/dt = −y_i / τ₋ + Σ_f δ(t − t_i^(f))  (19.13)

Each trace jumps by one at the moment of a spike and decays exponentially afterwards. The weight update then reads:

dw_ij/dt = A₋(w_ij)·y_i(t)·Σ_f δ(t − t_j^(f)) + A₊(w_ij)·x_j(t)·Σ_f δ(t − t_i^(f))  (19.14)

The behaviour is easier to follow in words. A trace acts as a fading record that a neuron fired recently. When a postsynaptic spike arrives, the weight is potentiated in proportion to whatever presynaptic trace remains. When a presynaptic spike arrives, the weight is depressed in proportion to whatever postsynaptic trace remains. The traces reproduce the exp(−|Δt|/τ±) factors of 19.10 exactly, which is why no spike history needs storing. Two extra variables per synapse are sufficient, and this is the main reason STDP is cheap to simulate and attractive for hardware. Either hard or soft bounds may be applied to A₊ and A₋.

19.2.3 Generalised STDP Models

Pair-based STDP does not account for all experimental results. It fails to reproduce the dependence of plasticity on how frequently spike pairs are repeated, and it fails on triplet and quadruplet protocols. Standard pair-based models predict that raising the repetition frequency should reduce net potentiation, whereas experiments show the opposite. They also predict that pre-post-pre and post-pre-post triplets give identical results, which is not observed.

The triplet rule of Pfister and Gerstner addresses this by giving the postsynaptic neuron two traces, a fast one y_{i,1} and a slow one y_{i,2}, with τ₁ < τ₂. Potentiation then depends on three spikes rather than two:

Δw₊ij(t_i^f) = A₊(w_ij) · x_j(t_i^f) · y{i,2}(t_i^{f−})  (19.15)

The notation t_i^{f−} indicates that y_{i,2} is read just before it is incremented by the current spike. Depression stays as in the pair-based rule. This model reproduces the observed frequency dependence and can be mapped onto the BCM rule, inheriting its input selectivity.

The Clopath model instead makes plasticity depend on the postsynaptic membrane potential rather than only on spikes. Potentiation requires the presynaptic trace to be nonzero, the momentary voltage to exceed a threshold θ₊, and a fast filtered voltage to exceed θ₋:

dw_ij^LTP/dt = +A_LTP · x̄_j(t) · [u_i(t) − θ₊]₊ · [ū_{i,+}(t) − θ₋]₊  if w_ij < w_max  (19.17)

The complete rule combines this with a depression term:

dw_ij/dt = −A_LTD(ū_i)·S_j(t)·[ū_{i,−}(t) − θ₋]₊ + A_LTP·x̄_j(t)·[u_i(t) − θ₊]₊·[ū_{i,+}(t) − θ₋]₊  (19.18)

with hard bounds 0 ≤ w_ij ≤ w_max. This reproduces experimental results that pair-based rules cannot explain.

19.3 Unsupervised Learning

The central claim of this section is that Hebbian plasticity tunes synaptic weights to the statistics of the input, without any external teaching signal.

The setup uses N input neurons and P input patterns. While pattern μ is presented, the input rates are ν_j = ξ_j^μ. The discrete form of the Hebb rule 19.3 is:

Δw_i = γ · ν^post · ν_i^pre  (19.21)

The small constant γ is the learning rate, related to the continuous-time coefficient by γ = c₁₁^corr · Δt for a pattern presented over a short interval Δt.

In a general rate model the postsynaptic rate passes through a nonlinear gain function:

ν^post = g( Σ_j w_j · ν_j^pre )  (19.22)

Restricting to a linear neuron simplifies the analysis considerably:

ν^post = Σ_j w_j · ν_j^pre = w · ν^pre  (19.23)

The output can therefore be read as the projection of the input vector onto the weight vector. Substituting 19.23 into 19.21 gives the weight change after presenting pattern ξ^μ:

Δw_i = γ Σ_j w_j · ν_j^pre · ν_i^pre = γ Σ_j w_j · ξ_j^μ · ξ_i^μ  (19.24)

which produces the iteration:

w_i(n+1) = w_i(n) + γ Σ_j w_j · ξ_j^{μn} · ξ_i^{μn}  (19.25)

This is an online rule, since the update follows immediately after each pattern.

19.3.1 and 19.3.2 Connection to Principal Component Analysis

Writing the expected evolution in matrix form, with C the correlation matrix and I the identity:

w(n+1) = (I + γC)·w(n) = (I + γC)^(n+1)·w(0)  (19.30)

Expressing the weight vector in terms of the eigenvectors e_k of C:

w(n) = Σ_k a_k(n)·e_k  (19.31)

gives an explicit solution:

w(n) = Σ_k (1 + λ_k)^n · a_k(0) · e_k  (19.32)

The correlation matrix is positive semi-definite, so every eigenvalue λ_k is real and non-negative. The weight vector grows exponentially, but growth is quickly dominated by the eigenvector with the largest eigenvalue, which is the first principal component:

w(n) → (1 + λ₁)^n · a₁(0) · e₁   as n → ∞  (19.33)

Since the output of the linear neuron is the projection of the input onto w, and w aligns with e₁, the neuron ends up reporting the projection onto the first principal component. Plain Hebbian learning therefore performs Principal Component Analysis, or PCA.

The first principal component points along the direction of maximum variance in the data. The variance of the projection onto an arbitrary unit direction y is:

σ²_y = ⟨[ξ^μ · y]²⟩_μ = y^T·V·y = Σ_i λ_i·a_i²  (19.37)

which is maximised when y = e₁.

Three limitations follow, each with a known fix:

LimitationFixThe PCA result holds for the expected weight vector, not the actual oneA sufficiently low learning rate keeps the actual vector close to the expected oneThe norm of the weight vector grows without boundOja's rule (19.7) normalises the length without changing directionPCA assumes data distributed around the origin, but firing rates are positiveThe covariance rule (19.6) works from deviations about a mean rate

19.3.3 Learning Equations for STDP Models

This subsection analyses the pair-based STDP rule under the assumption that spike trains are Poisson. The postsynaptic neuron uses a linear Poisson model with rate:

ν_i(u_i) = [α·u_i − ν₀]₊  (19.38)

where u_i(t) = Σ_j w_ij·ε(t − t_j^f) and ε(t) is the time course of an excitatory postsynaptic potential. The expected postsynaptic rate is then:

⟨ν_i⟩ = −ν₀ + α·ε̄·Σ_j w_ij·ν_j  (19.39)

with ε̄ = ∫ε(s)ds the area under the postsynaptic potential. Conditioned on an input spike at time t_j^f, the rate becomes:

ν_i(t) = −ν₀ + α·ε̄·Σ_j w_ij·ν_j + α·w_ij·ε(t − t_j^f)  (19.40)

Because the conditional rate differs from the expected rate, the output spike train is correlated with the input trains:

Γ_ji(s) = ⟨S_i(t+s)·S_j(t)⟩ = ⟨ν_i⟩·ν_j + α·w_ij·ν_j·ε(s)  (19.41)

The resulting expected weight evolution is:

⟨ẇ_ij⟩ = ν_j·⟨ν_i⟩·[ −A₋(w_ij)·τ₋ + A₊(w_ij)·τ₊ ] + α·w_ij·ν_j·A₊(w_ij)·∫W₊(s)·ε(s) ds  (19.42)

The first term resembles rate-based Hebbian learning, with a coefficient proportional to the integral under the learning window. The second term arises from pre-before-post spike timings and has no equivalent in a rate model.

If all input rates are equal, all weights converge to a common value w and the equation reduces to:

ẇ·ν = ν_i·[ −A₋(w)·τ₋ + A₊(w)·τ₊ ] + α·w·A₊(w)·∫W₊(s)·ε(s) ds  (19.43)

with the postsynaptic rate expressible as:

ν_i = −ν₀ + α·ν·ε̄·N·w  (19.44)

Defining C = A₋(w)·τ₋ − A₊(w)·τ₊ and C_ss = A₊(w)·∫W₊(s)·ε(s)ds, a fixed point exists when the window integral is negative, meaning depression dominates:

ν_FP = C_ss·ν₀ / (N·C·ν·ε̄ − C_ss)  (19.45)

This is an important practical result. When LTD dominates the learning window, STDP stabilises the postsynaptic firing rate at a fixed point, and since the output rate depends on the summed weights, stabilising the rate automatically normalises Σ_j w_ij. Competition between synapses therefore comes for free, without adding an explicit normalisation step.

19.4 Reward-Based Learning

Unsupervised learning tunes a network to input statistics but cannot learn a specific behaviour, such as pressing a button to obtain a reward. Doing so requires a third signal.

The Hebbian term H no longer changes the weight directly. Instead it sets an eligibility trace, which marks a synapse as recently co-active:

de_ij/dt = −e_ij/τ_e + H(pre_j , post_i)  (19.46)

When joint activity stops, H vanishes and the trace decays with time constant τ_e. The Hebbian term H may be any of the rate rules from 19.2.1 or an STDP rule such as 19.10.

An actual weight change requires both a nonzero eligibility trace and a neuromodulatory success signal M:

dw_ij/dt = M · H(pre_j , post_i) · e_ij  (19.47)

Standard Hebbian learning depends on two factors, presynaptic and postsynaptic activity. Here three factors are needed, so these are called three-factor learning rules. The modulator is usually taken as reward relative to expectation:

M(t) = R(t) − ⟨R⟩  (19.48)

where R is the reward and ⟨R⟩ is a running average. The eligibility time constant τ_e is typically around one second, chosen to bridge the delay between an action and the reward that follows it. Dopamine is the biological candidate for M, and experiments confirm that the STDP window shape is modulated by dopamine and other neuromodulators.

Relevance to This Project

The rule implemented here is the pair-based STDP model of 19.2.2, using the local variable form of equations 19.12 to 19.14 with hard bounds. This choice is justified on three grounds. It is the implementation the textbook recommends, it requires only two additional state variables per synapse, and it captures the timing asymmetry that motivates STDP in the first place. The triplet and Clopath rules are more accurate against experimental data, but the extra fidelity is not needed for a two class toy problem and would cost implementation time better spent elsewhere.

Amplitudes are chosen with A₋ slightly larger than A₊ so that depression dominates the window integral. Section 19.3.3 indicates this should produce the stabilising fixed point of equation 19.45, which limits runaway weight growth without an explicit normalisation step.

Winner-take-all competition is added on top, so that only the output neuron with the highest spike count updates its incoming weights on a given trial. This is a simplified stand-in for lateral inhibition and prevents both output neurons from learning the same input pattern.

Reward-based learning from 19.4 is not implemented. The toy classification task provides no reward signal. It is worth noting, however, that unsupervised STDP has no way of knowing which output neuron should correspond to which class label, so a learned solution may be correct but with labels swapped. Resolving that ambiguity is precisely what the third factor in equation 19.47 exists to do, which makes it a reasonable direction for future work.


In [12]:
#applying pair - based STDP - what the book recommends

#STDP parameters

tau_plus  = 20e-3   # presynaptic trace decay (LTP window width), 
tau_minus = 20e-3   # postsynaptic trace decay (LTD window width)
A_plus    = 0.010   # LTP amplitude  (A_+ > 0)
A_minus   = 0.012   # LTD amplitude  (A_- magnitude; slightly > A_plus so LTD
                    # dominates the window integral -> 19.3.3 fixed point ->
                    # self-stabilising rates, stops runaway growth)
w_max     = 2.0     # hard bound (Eq. 19.4 with beta -> 0)

w_sum_max = 2.0 #all weighted inputs will addd to this - hopefully preventing the infinite strenghtening of weights 

def run_trial_traced(x, W, max_rate, T, dt, V_rest, V_reset, V_th, R, tau,
                     t_refr, gbar, tau_syn, E_syn, rng):
    """
    replacing run_trial function which also returns raw input spike array, to apply STDP

    """
    
    spikes_in, rates = poisson_encode_population(x, max_rate, T, dt, rng)
    g_out = combined_conductance(spikes_in, W, gbar, tau_syn, dt)
    V_all, spikes_all = lif_conductance_population(
        g_out, dt, V_rest, V_reset, V_th, R, tau, t_refr, E_syn
    )
    spike_counts = np.array([len(s) for s in spikes_all])
    predicted_class = np.argmax(spike_counts)
    return spikes_in, spike_counts, spikes_all, predicted_class, V_all

def spike_times_to_array(spikes_all, n_steps, dt):
    """Convert ragged spike-time lists -> (N_out, n_steps) binary array."""
    N_out = len(spikes_all)
    S_out = np.zeros((N_out, n_steps))
    for o in range(N_out):
        for t_spike in spikes_all[o]:
            k = int(round(t_spike / dt))
            if 0 <= k < n_steps:
                S_out[o, k] = 1.0
    return S_out


def stdp_update(W, spikes_in, S_out, dt, winner=None):
    """
    Pair-based STDP via local traces -- direct implementation of Eq. 19.12-19.14.

    winner : if given (WTA), ONLY this output neuron's incoming weights change.
             Set to None to update all outputs (no competition).

    Returns the updated weight matrix (hard-bounded to [0, w_max]).
    """
    N_in, n_steps = spikes_in.shape
    N_out = W.shape[1]

    x = np.zeros(N_in)    # presynaptic traces  x_j   (Eq. 19.12)
    y = np.zeros(N_out)   # postsynaptic traces y_i   (Eq. 19.13)
    W = W.copy()   # work on a copy; apply changes as we go so soft bounds see current w

    decay_x = np.exp(-dt / tau_plus)
    decay_y = np.exp(-dt / tau_minus)

    outs = range(N_out) if winner is None else [winner]

    for k in range(n_steps):
        # --- traces decay one step (the -x/tau_+ part of 19.12) ---
        x *= decay_x
        y *= decay_y

        pre_now  = spikes_in[:, k] > 0
        post_now = S_out[:, k] > 0

        # --- Eq. 19.14, term 2: LTP at each POSTsynaptic spike,
        #     proportional to the lingering PREsynaptic trace x_j ---
        for o in outs:
            if post_now[o]:
                dW[:, o] += A_plus * x

        # --- Eq. 19.14, term 1: LTD at each PREsynaptic spike,
        #     proportional to the lingering POSTsynaptic trace y_i ---
        if pre_now.any():
            for o in outs:
                dW[pre_now, o] -= A_minus * y[o]

        x[pre_now]  += 1.0
        y[post_now] += 1.0

    return np.clip(W+dW, 0.0, w_max)



In [13]:
#now I will train the network with random weights

rng = np.random.default_rng(7)
W_learn = rng.uniform(0.4, 0.6, size=(2, 2)) #i made it close to symmetry to see what happens

print("initial W:\n", W_learn)

n_train = 200
n_steps = int(round(0.5 / dt))

for trial in range(n_train):
    true_class = trial % 2
    x = make_toy_sample(true_class, rng)

    spikes_in, spike_counts, spikes_all, pred, V_all = run_trial_traced(
        x, W_learn, max_rate=100, T=0.5, dt=dt,
        V_rest=V_rest, V_reset=V_reset, V_th=V_th, R=R, tau=tau, t_refr=t_refr,
        gbar=1e-7, tau_syn=5e-3, E_syn=0.0, rng=rng
    )

    S_out = spike_times_to_array(spikes_all, n_steps, dt)

    # WTA: only the neuron that fired most gets to learn this trial.
    # If nothing fired at all, skip no winner, nothing to reinforce.
    if spike_counts.sum() == 0:
        continue
    winner = int(np.argmax(spike_counts))

    W_learn = stdp_update(W_learn, spikes_in, S_out, dt, winner=winner)

    if trial % 40 == 0:
        print(f"trial {trial}: counts={spike_counts}, winner={winner}, "
              f"W=\n{np.round(W_learn, 3)}")

print("\nfinal learned W:\n", np.round(W_learn, 3))


initial W:
 [[0.52501909 0.57944276]
 [0.55513714 0.44504144]]
trial 0: counts=[ 7 10], winner=1, W=
[[0.525 0.673]
 [0.555 0.431]]
trial 40: counts=[42 58], winner=1, W=
[[0.901 2.   ]
 [2.    0.494]]
trial 80: counts=[43 65], winner=1, W=
[[0.938 2.   ]
 [2.    0.579]]
trial 120: counts=[62 71], winner=1, W=
[[1.399 2.   ]
 [2.    1.025]]
trial 160: counts=[79 72], winner=0, W=
[[2.    2.   ]
 [2.    1.266]]

final learned W:
 [[2.    2.   ]
 [2.    1.266]]


In [14]:
# --- Test learned weights on fresh trials ---
rng_test = np.random.default_rng(99)
correct = 0
n_test = 40
for trial in range(n_test):
    true_class = trial % 2
    x = make_toy_sample(true_class, rng_test)
    spike_counts, spikes_all, pred = run_trial(
        x, W_learn, max_rate=100, T=0.5, dt=dt,
        V_rest=V_rest, V_reset=V_reset, V_th=V_th, R=R, tau=tau, t_refr=t_refr,
        gbar=1e-7, tau_syn=5e-3, E_syn=0.0, rng=rng_test
    )
    if pred == true_class:
        correct += 1
print(f"learned-weight accuracy: {correct}/{n_test} = {100*correct/n_test:.1f}%")

learned-weight accuracy: 20/40 = 50.0%
